# Continuous VP Diffusion

This notebook uses the real [`ContinuousVPDiffusion`](./continuous.py) class together with the real `CSPTask` and `DNGTask` dataloaders.

The goal is to make the training idea easy to follow:

1. Start from clean data `x0`
2. Sample a random diffusion time `t`
3. Create a noisy version `x_t`
4. Compute the exact target score
5. Train a model to predict that target

## Theory

For the VP forward process we use the closed-form Gaussian marginal

$$
x_t \mid x_0 \sim \mathcal{N}(\alpha(t)x_0, \sigma(t)^2 I)
$$

so we can sample directly with

$$
x_t = \alpha(t)x_0 + \sigma(t)\epsilon, \quad \epsilon \sim \mathcal{N}(0, I)
$$

and the exact conditional score target is

$$
\nabla_{x_t} \log p(x_t \mid x_0) = -\epsilon / \sigma(t)
$$

That means training becomes ordinary supervised learning on noisy inputs.

In [ ]:
from pathlib import Path

import torch
from torch import nn

from kldm.data import CSPTask, DNGTask, resolve_data_root
from kldm.diffusionModels.continuous import ContinuousVPDiffusion

torch.manual_seed(7)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
root = resolve_data_root()

diffusion = ContinuousVPDiffusion().to(device)
diffusion

## 1. Load Real CSP And DNG Batches

We use the real MatterGen-backed loaders:

- `CSPTask` gives a graph-level lattice representation in `batch.l`
- `DNGTask` gives node-level one-hot atom features in `batch.h`

In [ ]:
csp_batch = next(iter(CSPTask().dataloader(root=root, split='train', batch_size=2, shuffle=False, download=True)))
dng_batch = next(iter(DNGTask().dataloader(root=root, split='train', batch_size=2, shuffle=False, download=True)))

csp_batch = csp_batch.to(device)
dng_batch = dng_batch.to(device)

print('CSP lattice shape:', tuple(csp_batch.l.shape))
print('CSP atom shape   :', tuple(csp_batch.h.shape))
print('DNG lattice shape:', tuple(dng_batch.l.shape))
print('DNG atom shape   :', tuple(dng_batch.h.shape))

## 2. CSP Case: Diffuse The Lattice

For CSP we treat the lattice vector

$$
l = [\log a, \log b, \log c, \tan(\alpha - \pi/2), \tan(\beta - \pi/2), \tan(\gamma - \pi/2)]
$$

as the clean Euclidean variable `x0`.

In [ ]:
csp_x0 = csp_batch.l
csp_t = torch.rand(csp_x0.shape[0], device=device)
csp_xt, csp_eps = diffusion.forward_sample(csp_t, csp_x0)
csp_target = diffusion.score_target(csp_t, csp_eps)

print('csp_t:', csp_t)
print('\nBefore diffusion (first structure):')
print(csp_x0[0])
print('\nNoise used:')
print(csp_eps[0])
print('\nAfter diffusion:')
print(csp_xt[0])
print('\nTarget score:')
print(csp_target[0])

## 3. DNG Case: Diffuse The Atom Features

For DNG we use the continuous one-hot atom representation in `batch.h`.

Important detail:

- `batch.h` is node-level, shape `(num_nodes, num_species)`
- diffusion time is sampled per graph
- then copied to each node with `batch.batch`

In [ ]:
dng_x0 = dng_batch.h
graph_t = torch.rand(dng_batch.num_graphs, device=device)
node_t = graph_t[dng_batch.batch]
dng_xt, dng_eps = diffusion.forward_sample(node_t, dng_x0)
dng_target = diffusion.score_target(node_t, dng_eps)

print('graph_t:', graph_t)
print('\nBefore diffusion (first 3 atoms):')
print(dng_x0[:3])
print('\nNoise used (first 3 atoms):')
print(dng_eps[:3])
print('\nAfter diffusion (first 3 atoms):')
print(dng_xt[:3])
print('\nTarget score (first 3 atoms):')
print(dng_target[:3])

## 4. Minimal Score Model

We now build the smallest possible score network for the lattice case.

The input is noisy data `x_t` together with time `t`. The network predicts a tensor with the same shape as `x_t`, which we compare to the exact score target.

In [ ]:
class TinyScoreNet(nn.Module):
    def __init__(self, feature_dim: int, hidden_dim: int = 64) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feature_dim + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, feature_dim),
        )

    def forward(self, x_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        if x_t.ndim == 2:
            return self.net(torch.cat([x_t, t[:, None]], dim=-1))

        t_embed = t[:, None]
        while t_embed.ndim < x_t.ndim:
            t_embed = t_embed.unsqueeze(1)
        t_embed = t_embed.expand(*x_t.shape[:-1], 1)
        return self.net(torch.cat([x_t, t_embed], dim=-1))


lattice_model = TinyScoreNet(feature_dim=6).to(device)
optimizer = torch.optim.Adam(lattice_model.parameters(), lr=1e-3)
lattice_model

## 5. One Training Step

This is the core diffusion training pattern:

1. sample clean data `x0`
2. sample random time `t`
3. build noisy `x_t`
4. compute the exact target score
5. train the model to predict that score

In [ ]:
batch = next(iter(CSPTask().dataloader(root=root, split='train', batch_size=8, shuffle=True, download=True))).to(device)

x0 = batch.l
t = torch.rand(x0.shape[0], device=x0.device)
x_t, eps = diffusion.forward_sample(t, x0)
target = diffusion.score_target(t, eps)

pred = lattice_model(x_t, t)
loss = ((pred - target) ** 2).mean()

optimizer.zero_grad()
loss.backward()
optimizer.step()

print('x0 shape    :', tuple(x0.shape))
print('x_t shape   :', tuple(x_t.shape))
print('target shape:', tuple(target.shape))
print('pred shape  :', tuple(pred.shape))
print('loss        :', float(loss))

## 6. The Training Idea In Loop Form

This is the same idea written as a small loop.

We are **not** simulating many small forward steps for each sample. Instead, for each batch we directly sample a random time `t` and jump to the corresponding noisy point `x_t` using the closed-form formula.

In [ ]:
train_loader = CSPTask().dataloader(root=root, split='train', batch_size=16, shuffle=True, download=True)

for step, batch in enumerate(train_loader):
    batch = batch.to(device)

    x0 = batch.l
    t = torch.rand(x0.shape[0], device=x0.device)
    x_t, eps = diffusion.forward_sample(t, x0)
    target = diffusion.score_target(t, eps)

    pred = lattice_model(x_t, t)
    loss = ((pred - target) ** 2).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f'step={step} loss={float(loss):.6f}')
    if step == 2:
        break

## 7. The Same Pattern For DNG Atoms

For atoms the only important difference is that the features are node-level, so `t` must also be node-level.

That gives us supervised pairs of the form

$$
(x_0, t) \rightarrow (x_t, \text{target})
$$

where `target` is the exact score of the Gaussian forward process.

In [ ]:
atom_model = TinyScoreNet(feature_dim=dng_batch.h.shape[-1]).to(device)
atom_optimizer = torch.optim.Adam(atom_model.parameters(), lr=1e-3)

batch = next(iter(DNGTask().dataloader(root=root, split='train', batch_size=4, shuffle=True, download=True))).to(device)

x0 = batch.h
graph_t = torch.rand(batch.num_graphs, device=x0.device)
t = graph_t[batch.batch]
x_t, eps = diffusion.forward_sample(t, x0)
target = diffusion.score_target(t, eps)

pred = atom_model(x_t, t)
loss = ((pred - target) ** 2).mean()

atom_optimizer.zero_grad()
loss.backward()
atom_optimizer.step()

print('atom batch shape :', tuple(x0.shape))
print('pred shape       :', tuple(pred.shape))
print('target shape     :', tuple(target.shape))
print('loss             :', float(loss))